In [1]:
import json, re
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_core.documents import Document
load_dotenv(find_dotenv(), override=True)          # LangSmith tracing stays on

def load_chunks(path):
    with open(path, encoding="utf-8") as f:
        return [Document(page_content=r["text"], metadata=r["metadata"]) for r in map(json.loads, f)]

CHUNKS = {
    "article": load_chunks("../data/chunks/article_chunks.jsonl"),
    "fixed":   load_chunks("../data/chunks/fixed_chunks.jsonl"),
}
for name, ch in CHUNKS.items():
    print(f"{name:8} {len(ch)} chunks")

article  204 chunks
fixed    262 chunks


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

emb = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",            # small, fast, runs on your Mac, free
    encode_kwargs={"normalize_embeddings": True},   # vectors of length 1 -> dot product = cosine similarity
)

sentences = [
    "How many days of annual leave does an employee get?",
    "The worker is entitled to thirty days of paid vacation each year.",
    "The controller must notify the data office of a personal data breach.",
]
vecs = np.array(emb.embed_documents(sentences))
print("Each text becomes", vecs.shape[1], "numbers")
print("question vs leave rule :", round(float(vecs[0] @ vecs[1]), 3))
print("question vs data breach:", round(float(vecs[0] @ vecs[2]), 3))

/Users/abdul/Projects/uae-labour-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1022 23:35:11.034000 55801 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20623.82it/s]


Each text becomes 384 numbers
question vs leave rule : 0.703
question vs data breach: 0.444


In [3]:
import shutil
from langchain_chroma import Chroma

CHROMA_DIR = "../data/chroma"
shutil.rmtree(CHROMA_DIR, ignore_errors=True)       # rebuild from scratch: no old/duplicate vectors

stores = {}
for name, chunks in CHUNKS.items():
    stores[name] = Chroma.from_documents(
        chunks, emb,
        collection_name=f"uae_{name}",
        persist_directory=CHROMA_DIR,                # saved to disk, so it can be reused without re-embedding
        ids=[c.metadata["chunk_id"] for c in chunks],
        collection_metadata={"hnsw:space": "cosine"},
    )
    print(f"{name:8} stored {stores[name]._collection.count()} vectors")

article  stored 204 vectors
fixed    stored 262 vectors


In [4]:
def label(d):                                        # short readable ID, e.g. "labour_law:29"
    return f"{d.metadata['source']}:{d.metadata['article']}"

q = "How many days of annual leave do I get?"
for d, score in stores["article"].similarity_search_with_relevance_scores(q, k=5):
    print(f"{score:.3f}  {label(d):28} {d.page_content[:70]!r}")

0.809  labour_law:29                '[Labour Law — Article 29, part 1/2]\nArticle (29)\nAnnual leave\n1. Witho'
0.782  labour_law:29                '[Labour Law — Article 29, part 2/2]\ndate for enjoying his leave at lea'
0.762  labour_law:32                '[Labour Law — Article 32]\nArticle (32)\nVarious holidays\n1. The Worker '
0.741  exec_regs:18                 '[Executive Regulation — Article 18]\nArticle (18)\nAnnual Leave of Worke'
0.716  labour_law:28                '[Labour Law — Article 28]\nArticle (28)\nOfficial Holidays and Requiring'


In [5]:
from langchain_community.retrievers import BM25Retriever

def tokenize(text):                                  # lowercase + split on non-letters: "Leave," -> "leave"
    return re.findall(r"\w+", text.lower())

bm25 = {name: BM25Retriever.from_documents(ch, k=10, preprocess_func=tokenize)
        for name, ch in CHUNKS.items()}

for d in bm25["article"].invoke("Article 51 gratuity")[:5]:
    print(f"{label(d):28} {d.page_content[:70]!r}")

labour_law:51                '[Labour Law — Article 51, part 2/2]\npiece-meal.\n6. The foregoing is pr'
labour_law:51                '[Labour Law — Article 51, part 1/2]\nArticle (51)\nEnd-of-Service Gratui'
labour_law:52                '[Labour Law — Article 52]\nArticle (52)\nEnd-of-Service Gratuity for Wor'
labour_law:68                '[Labour Law — Article 68]\nArticle (68)\nRegularization\n1. The provision'
labour_law:47                "[Labour Law — Article 47]\nArticle (47)\nUnlawful Termination of Worker'"


/var/folders/0w/lhff5nrj5jgc5m4mkwnkg5g00000gn/T/ipykernel_55801/3792166628.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [6]:
def rrf(result_lists, k=60, top_n=5):
    """Merge ranked lists. Each chunk gets 1/(k + rank) from every list it appears in.
    Being near the top of BOTH lists wins. Uses ranks only, so the two different score scales don't matter."""
    scores, docs = {}, {}
    for results in result_lists:
        for rank, d in enumerate(results, 1):
            cid = d.metadata["chunk_id"]
            scores[cid] = scores.get(cid, 0) + 1 / (k + rank)
            docs[cid] = d
    best = sorted(scores, key=scores.get, reverse=True)[:top_n]
    return [(docs[c], scores[c]) for c in best]

def hybrid_search(query, name="article", top_n=5):
    dense  = stores[name].similarity_search(query, k=10)
    sparse = bm25[name].invoke(query)
    return rrf([dense, sparse], top_n=top_n)

for d, s in hybrid_search("How many days of annual leave do I get?"):
    print(f"{s:.4f}  {label(d):28} {d.page_content[:70]!r}")

0.0325  labour_law:29                '[Labour Law — Article 29, part 1/2]\nArticle (29)\nAnnual leave\n1. Witho'
0.0325  labour_law:29                '[Labour Law — Article 29, part 2/2]\ndate for enjoying his leave at lea'
0.0312  exec_regs:18                 '[Executive Regulation — Article 18]\nArticle (18)\nAnnual Leave of Worke'
0.0304  labour_law:32                '[Labour Law — Article 32]\nArticle (32)\nVarious holidays\n1. The Worker '
0.0302  exec_regs:19                 '[Executive Regulation — Article 19]\nArticle (19)\nCarrying Forward the '


In [7]:
probes = [
    "How many days of annual leave do I get?",             # meaning
    "What does Article 51 say?",                           # exact number
    "My employer has not paid my salary for two months",   # paraphrase, no legal words
    "end of service gratuity calculation",                 # legal keywords
    "Can my company share my personal data with others?",  # other law (PDPL)
]
top3 = lambda docs: ", ".join(label(d) for d in docs[:3])
for q in probes:
    print(f"\nQ: {q}")
    print("  dense :", top3(stores["article"].similarity_search(q, k=3)))
    print("  bm25  :", top3(bm25["article"].invoke(q)))
    print("  hybrid:", top3([d for d, _ in hybrid_search(q)]))


Q: How many days of annual leave do I get?
  dense : labour_law:29, labour_law:29, labour_law:32
  bm25  : labour_law:29, labour_law:29, exec_regs:19
  hybrid: labour_law:29, labour_law:29, exec_regs:18

Q: What does Article 51 say?
  dense : labour_law_amendment_2022:1, exec_regs:38, pdpl:23
  bm25  : labour_law:51, labour_law:51, exec_regs:19
  hybrid: labour_law_amendment_2022:1, labour_law:51, exec_regs:38

Q: My employer has not paid my salary for two months
  dense : labour_law:9, labour_law:40, labour_law:47
  bm25  : labour_law:30, labour_law:32, labour_law:54
  hybrid: labour_law:9, labour_law:30, labour_law:40

Q: end of service gratuity calculation
  dense : labour_law:51, labour_law:51, exec_regs:30
  bm25  : labour_law:51, labour_law:51, labour_law:52
  hybrid: labour_law:51, labour_law:51, labour_law:52

Q: Can my company share my personal data with others?
  dense : pdpl:22, pdpl:23, pdpl:14
  bm25  : pdpl:1, pdpl:1, pdpl:1
  hybrid: pdpl:22, pdpl:5, pdpl:7


OPTIONAL TO TEST

In [8]:
by_id = {c.metadata["chunk_id"]: c for c in CHUNKS["article"]}
for cid in ["labour_law-a9-p1", "labour_law-a30-p1", "labour_law-a40-p1", "labour_law_amendment_2022-a1-p1"]:
    print(cid, "->", by_id[cid].page_content[:160].replace("\n", " | "), "\n")

# Which Labour Law articles are about wages? (title = 3rd line of the chunk)
print([label(c) + " " + c.page_content.splitlines()[2] for c in CHUNKS["article"]
       if c.metadata["source"] == "labour_law" and "wage" in c.page_content.splitlines()[2].lower()])

labour_law-a9-p1 -> [Labour Law — Article 9, part 1/2] | Article (9) | Probation Period | 1. The Employer may appoint the Worker under the Probation Period for a period not | exceeding six 

labour_law-a30-p1 -> [Labour Law — Article 30, part 1/2] | Article (30) | Maternity Leave | 1. The female Worker shall be entitled to maternity leave of (60) sixty days, according to the |  

labour_law-a40-p1 -> [Labour Law — Article 40] | Article (40) | Suspension of Work | 1. The Employer may suspend the Worker temporarily from work for a period not exceeding | thirty (30) da 

labour_law_amendment_2022-a1-p1 -> [Labour Law Amendment 2022 — Article 1] | Article (1) | The text o f clause (3) of Article (8) of Federal Decree by Law No. (33) of 2021 Regulating | Labour Relations 

['labour_law:22 Determining the Amount or Type of Wage and Paying It', 'labour_law:23 Method of Calculating Wages for Workers on Piecemeal Basis', 'labour_law:24 Transferring Monthly Wage Worker to Other Categories', "lab